# Daily Challenge: Stock Price Prediction with LSTM (PyTorch)

**Course:** Developers Institute  **Week 6 - Day 3**  
**Author:** Alex Goldbaum

End-to-end PyTorch pipeline: fetch a real stock series, preprocess it for a
supervised forecasting task, build a custom `Dataset` + `DataLoader`, train an
**LSTM** to predict tomorrow's closing price from the last *N* days, evaluate
with R² and persist the model + scaler for future inference.


## 1. Install Required Libraries


In [ ]:
%pip install -qU torch scikit-learn yfinance joblib


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error

import yfinance as yf
import joblib

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## 2. Load and Preprocess the Dataset

We download ~5 years of **AAPL** daily price history with `yfinance` (no
auth needed). We then drop the columns we don't use, build the **next-day
Close** as the prediction target, and scale all features to `[0, 1]`.


In [ ]:
TICKER = 'AAPL'
raw = yf.download(TICKER, period='5y', interval='1d', progress=False, auto_adjust=False)

# Flatten possible MultiIndex columns (newer yfinance versions)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = [c[0] for c in raw.columns]

print(f'Downloaded {len(raw)} rows of {TICKER} data from {raw.index[0].date()} to {raw.index[-1].date()}')
raw.head()


In [ ]:
# Drop unnecessary columns. We keep OHLCV (Open, High, Low, Close, Volume).
df = raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

# Target: NEXT day's Close
df['Target'] = df['Close'].shift(-1)
df = df.dropna()

print('Shape after target construction:', df.shape)
df.tail()


In [ ]:
# Quick visualization of the close price
plt.figure(figsize=(13, 4.5))
plt.plot(df.index, df['Close'], color='steelblue', linewidth=1)
plt.title(f'{TICKER} closing price', fontweight='bold')
plt.xlabel('Date'); plt.ylabel('Close (USD)')
plt.tight_layout()
plt.show()


In [ ]:
# Normalize features and target separately so we can inverse-transform the prediction
feature_cols = ['Open', 'High', 'Low', 'Close', 'Volume']

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

X_raw = df[feature_cols].values
y_raw = df[['Target']].values

X_scaled = feature_scaler.fit_transform(X_raw)
y_scaled = target_scaler.fit_transform(y_raw).ravel()

print('Feature range:', X_scaled.min().round(3), '->', X_scaled.max().round(3))
print('Target range :', y_scaled.min().round(3), '->', y_scaled.max().round(3))


## 3. Prepare the Dataset for Training

We build a custom `torch.utils.data.Dataset` that turns the time series into
sliding windows of `seq_len` days. Then we split chronologically (no
shuffling — the future must come *after* the past) into train / val / test.


In [ ]:
SEQ_LEN = 30  # use the last 30 days to predict the next day


class StockDataset(Dataset):
    def __init__(self, X, y, seq_len):
        self.X = X
        self.y = y
        self.seq_len = seq_len

    def __len__(self):
        return len(self.X) - self.seq_len

    def __getitem__(self, idx):
        x_seq = self.X[idx:idx + self.seq_len]
        y_val = self.y[idx + self.seq_len - 1]
        return (
            torch.tensor(x_seq, dtype=torch.float32),
            torch.tensor(y_val, dtype=torch.float32),
        )


In [ ]:
# Chronological 70 / 15 / 15 split
n = len(X_scaled)
train_end = int(0.70 * n)
val_end = int(0.85 * n)

X_train, X_val, X_test = X_scaled[:train_end], X_scaled[train_end:val_end], X_scaled[val_end:]
y_train, y_val, y_test = y_scaled[:train_end], y_scaled[train_end:val_end], y_scaled[val_end:]

train_ds = StockDataset(X_train, y_train, SEQ_LEN)
val_ds = StockDataset(X_val, y_val, SEQ_LEN)
test_ds = StockDataset(X_test, y_test, SEQ_LEN)

print(f'Train rows: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


## 4. Define the LSTM Model

`nn.LSTM → Dropout → Linear`. We take the **last hidden state** of the LSTM
as the summary of the input sequence and map it to a single scalar (next-day
scaled price).


In [ ]:
class StockLSTM(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, seq_len, n_features)
        out, _ = self.lstm(x)
        last = out[:, -1, :]  # last timestep summary
        last = self.dropout(last)
        return self.fc(last).squeeze(-1)  # (batch,)


n_features = X_scaled.shape[1]
model = StockLSTM(n_features=n_features, hidden_size=64, num_layers=2, dropout=0.2).to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal trainable params: {n_params:,}')


## 5. Train the Model

Adam optimizer, MSE loss, and a manual train/validation loop. We keep the
best-validation-loss weights with a simple early-stop bookkeeping.


In [ ]:
EPOCHS = 30
LR = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()

history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
best_state = None
patience = 5
stale = 0

for epoch in range(1, EPOCHS + 1):
    # ---- train ----
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    # ---- validate ----
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            val_loss += criterion(pred, yb).item() * xb.size(0)
    val_loss /= len(val_loader.dataset)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    print(f'Epoch {epoch:02d}  train_loss={train_loss:.5f}  val_loss={val_loss:.5f}')

    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stale = 0
    else:
        stale += 1
        if stale >= patience:
            print(f'Early stopping at epoch {epoch} (no val_loss improvement for {patience} epochs).')
            break

if best_state is not None:
    model.load_state_dict(best_state)
print(f'\nBest val loss: {best_val_loss:.5f}')


In [ ]:
# Training curves
plt.figure(figsize=(11, 4.5))
plt.plot(history['train_loss'], label='train loss')
plt.plot(history['val_loss'], label='val loss')
plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.title('LSTM training — loss across epochs', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


## 6. Evaluate the Model

We score the test split with **R²** (proportion of variance explained) and
RMSE (in USD, after inverse-transforming the predictions).


In [ ]:
model.eval()
preds, trues = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        pred = model(xb).cpu().numpy()
        preds.append(pred)
        trues.append(yb.numpy())

y_pred_scaled = np.concatenate(preds)
y_true_scaled = np.concatenate(trues)

# Inverse-transform back to USD
y_pred = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
y_true = target_scaler.inverse_transform(y_true_scaled.reshape(-1, 1)).ravel()

r2 = r2_score(y_true, y_pred)
rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
mae = float(np.mean(np.abs(y_pred - y_true)))
print(f'Test R^2 : {r2:.4f}')
print(f'Test RMSE: ${rmse:.3f}')
print(f'Test MAE : ${mae:.3f}')


In [ ]:
# Plot predicted vs actual on the test set
plt.figure(figsize=(13, 4.5))
plt.plot(y_true, label='Actual', color='steelblue', linewidth=1.4)
plt.plot(y_pred, label='Predicted', color='tomato', linewidth=1.4, alpha=0.85)
plt.title(f'{TICKER} — predicted vs actual closing price (test set)', fontweight='bold')
plt.xlabel('Test day index')
plt.ylabel('Close (USD)')
plt.legend()
plt.tight_layout()
plt.show()

# Scatter for a more rigorous look
plt.figure(figsize=(6, 6))
plt.scatter(y_true, y_pred, alpha=0.6, edgecolor='white', color='steelblue')
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
plt.plot(lims, lims, 'k--', alpha=0.5)
plt.xlim(lims); plt.ylim(lims)
plt.xlabel('Actual close (USD)')
plt.ylabel('Predicted close (USD)')
plt.title(f'Predicted vs actual — R^2 = {r2:.3f}', fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Persist the Model and Scalers

Save the trained PyTorch weights with `torch.save` and the two scalers with
`joblib`. These three artifacts are all we need to run inference later on new
unseen data.


In [ ]:
import pathlib
out_dir = pathlib.Path('/content/stock_lstm_artifacts')
out_dir.mkdir(parents=True, exist_ok=True)

torch.save(model.state_dict(), out_dir / 'lstm_weights.pt')
joblib.dump(feature_scaler, out_dir / 'feature_scaler.joblib')
joblib.dump(target_scaler, out_dir / 'target_scaler.joblib')

# Save config so we can rebuild the same architecture later
config = {
    'ticker': TICKER,
    'seq_len': SEQ_LEN,
    'feature_cols': feature_cols,
    'n_features': n_features,
    'hidden_size': 64,
    'num_layers': 2,
    'dropout': 0.2,
}
import json as _json
with open(out_dir / 'config.json', 'w') as f:
    _json.dump(config, f, indent=2)

print('Saved:')
for p in out_dir.iterdir():
    print(' -', p.name, f'({p.stat().st_size / 1024:.1f} KB)')


In [ ]:
# Sanity check: reload everything and confirm we get the same predictions
feature_scaler_loaded = joblib.load(out_dir / 'feature_scaler.joblib')
target_scaler_loaded = joblib.load(out_dir / 'target_scaler.joblib')

model_loaded = StockLSTM(n_features=n_features, hidden_size=64, num_layers=2, dropout=0.2).to(device)
model_loaded.load_state_dict(torch.load(out_dir / 'lstm_weights.pt'))
model_loaded.eval()

# Predict on the first test batch with both models
with torch.no_grad():
    xb, _ = next(iter(test_loader))
    p_orig = model(xb.to(device)).cpu().numpy()
    p_load = model_loaded(xb.to(device)).cpu().numpy()

print('Predictions match after reload:', np.allclose(p_orig, p_load))


## 8. Discussion

**What R² means here.** R² is the proportion of variance in the test target
that the model explains. A value close to 1.0 looks great — but on stock data
this is easy to misread: because the predicted sequence is shifted by one day,
the model can score high simply by *almost copying yesterday's price*. Always
compare against a naive baseline (predict-yesterday) before claiming the model
is genuinely forecasting.

**Limitations.**
- We trained on **OHLCV** only. Real forecasting systems add macro signals,
  fundamentals, news embeddings, options flow, etc.
- A single seed and one ticker is not a serious evaluation. Cross-validate
  across tickers and time windows before drawing conclusions.
- **Non-stationarity.** The distribution of returns changes over time (regime
  changes, COVID, rate cycles). A model trained on 2019-2023 can fail in 2024.
  Re-train regularly and monitor live performance.
- **Markets are nearly efficient.** Beating a simple baseline by a meaningful
  margin is *much* harder than it looks. Don't trade on this notebook.

**Next steps.**
1. Predict **returns** (log returns) instead of absolute price — stationarity is much better.
2. Compare against `predict_yesterday` and `predict_mean` baselines.
3. Add technical features (RSI, MACD, moving averages) as extra inputs.
4. Try a Transformer encoder for the same windowed input — same data,
   different inductive bias.
